# EXP1 — Верификация (pilot, день 1)

Прогон 8 существующих сценариев из `code/onto/ontologies/scenarios/` через `VerificationService` и сбор первой confusion matrix по пяти свойствам СВ-1..СВ-5.

Pilot нужен как sanity-проверка методологии: паттерн прогона, извлечения статусов, расчёта P/R/F1. На 8 точках метрики не репрезентативны — репрезентативная выборка из 80–100 сценариев генерируется в день 2 через `_common/generator.py`.

Ground truth и сценарии — те же, что в `code/backend/src/tests/integration/test_verification_scenarios.py`.

In [1]:
import json
import os
import sys
import time
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
EXPERIMENTS_ROOT = NOTEBOOK_DIR.parent
PROJECT_ROOT = EXPERIMENTS_ROOT.parent
BACKEND_SRC = PROJECT_ROOT / 'code' / 'backend' / 'src'
SCENARIOS_OWL_DIR = PROJECT_ROOT / 'code' / 'onto' / 'ontologies' / 'scenarios'
GROUND_TRUTH_PATH = BACKEND_SRC / 'tests' / 'fixtures' / 'scenarios_ground_truth.json'
RESULTS_DIR = NOTEBOOK_DIR / 'results'
PZ_FIGURES_DIR = PROJECT_ROOT / 'pz' / 'figures' / 'exp1'

sys.path.insert(0, str(BACKEND_SRC))
sys.path.insert(0, str(EXPERIMENTS_ROOT))

from owlready2 import World

from services.ontology_core import OntologyCore
from services.cache_manager import CacheManager
from services.reasoning import ReasoningOrchestrator
from services.verification import VerificationService

from _common.metrics import (
    aggregate_by_property,
    format_csv,
    format_markdown_table,
    macro_average,
)

print('BACKEND_SRC :', BACKEND_SRC)
print('SCENARIOS   :', SCENARIOS_OWL_DIR)
print('GROUND_TRUTH:', GROUND_TRUTH_PATH)

BACKEND_SRC : C:\Documents\itmo\vkr\vkr-access-control\code\backend\src
SCENARIOS   : C:\Documents\itmo\vkr\vkr-access-control\code\onto\ontologies\scenarios
GROUND_TRUTH: C:\Documents\itmo\vkr\vkr-access-control\code\backend\src\tests\fixtures\scenarios_ground_truth.json


In [2]:
with GROUND_TRUTH_PATH.open(encoding='utf-8') as f:
    ground_truth = json.load(f)['scenarios']

print(f'Сценариев в ground truth: {len(ground_truth)}')
for spec in ground_truth:
    expected_props = [k for k, v in spec['expected'].items() if isinstance(v, str)]
    full = spec.get('full', False)
    print(f"  {spec['name']:35s}  full={str(full):5s}  props={expected_props}")

Сценариев в ground truth: 8
  happy_path                           full=True   props=['consistency', 'acyclicity', 'reachability', 'redundancy', 'subsumption']
  bad_sv1_disjointness                 full=False  props=['consistency']
  bad_sv2_cycle                        full=False  props=['consistency', 'acyclicity']
  bad_sv3_atomic_threshold             full=False  props=['consistency', 'acyclicity', 'reachability']
  bad_sv3_empty_date                   full=False  props=['consistency', 'acyclicity', 'reachability']
  bad_sv3_structural                   full=False  props=['consistency', 'acyclicity', 'reachability']
  bad_sv4_redundant                    full=True   props=['consistency', 'acyclicity', 'reachability', 'redundancy']
  bad_sv5_subject                      full=True   props=['consistency', 'acyclicity', 'reachability', 'subsumption']


In [3]:
def run_scenario(spec: dict) -> tuple[object, float]:
    owl_path = SCENARIOS_OWL_DIR / f"{spec['name']}.owl"
    world = World()
    onto = world.get_ontology(f"file://{str(owl_path).replace(os.sep, '/')}").load()
    core = OntologyCore(onto_path=str(owl_path), world=world)
    core.onto = onto
    core.world = world
    reasoner = ReasoningOrchestrator(core.onto)
    cache = CacheManager(None)
    service = VerificationService(core, reasoner=reasoner, cache=cache)

    t0 = time.perf_counter()
    report = service.verify(spec['course_id'], include_subsumption=spec.get('full', False))
    duration_ms = (time.perf_counter() - t0) * 1000.0
    return report, duration_ms


runs: list[tuple[dict, object, float]] = []
for spec in ground_truth:
    report, duration_ms = run_scenario(spec)
    runs.append((spec, report, duration_ms))
    status_summary = ', '.join(
        f"{p}={report.properties[p].status}" for p in report.properties
    )
    print(f"{spec['name']:35s}  {duration_ms:7.1f} ms  {status_summary}")

* Owlready2 * Running Pellet...
    java -Xmx2000M -cp C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-runtime-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\aterm-java-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\commons-codec-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpclient-4.2.3.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpcore-4.2.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\jcl-over-slf4j-1.6.4.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\jena-arq-fixed2.10.0.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib

* Owlready2 * Pellet took 1.361236810684204 seconds
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)
* Owlready2 * Running Pellet...
    java -Xmx2000M -cp C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-runtime-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\aterm-java-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\commons-codec-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpclient-4.2.3.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpcore-4.2.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\jcl-over-slf4j-1.6.4.jar;C:\Users\ga

* Owlready * Adding relation happy_path.seasonal_workshop is_available_for happy_path.student_sidorov
* Owlready * Adding relation happy_path.seasonal_workshop is_available_for happy_path.student_korolev
* Owlready * Adding relation happy_path.seasonal_workshop is_available_for happy_path.student_petrov
* Owlready * Adding relation happy_path.seasonal_workshop is_available_for happy_path.student_sandbox
* Owlready * Adding relation happy_path.seasonal_workshop is_available_for happy_path.student_ivanov
* Owlready * Adding relation happy_path.student_petrov satisfies happy_path.p3_quiz1_requires_viewed_lecture1
* Owlready * Adding relation happy_path.student_petrov satisfies happy_path.p5_seasonal_workshop_date_window
* Owlready * Adding relation happy_path.student_sidorov satisfies happy_path.p6_sub_b_quiz3_grade70
* Owlready * Adding relation happy_path.student_sidorov satisfies happy_path.p7_sub_a_comp_basic_syntax
* Owlready * Adding relation happy_path.student_sidorov satisfies hap

Онтология inconsistent: Java error message is: ERROR: Ontology is inconsistent, run "pellet explain" to get the reason



* Owlready2 * Running Pellet...
    java -Xmx2000M -cp C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-runtime-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\aterm-java-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\commons-codec-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpclient-4.2.3.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpcore-4.2.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\jcl-over-slf4j-1.6.4.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\jena-arq-fixed2.10.0.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib

...

bad_sv3_structural                    1051.3 ms  consistency=passed, acyclicity=failed, reachability=failed


* Owlready2 * Pellet took 1.0238761901855469 seconds
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)
* Owlready2 * Running Pellet...
    java -Xmx2000M -cp C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-runtime-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\aterm-java-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\commons-codec-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpclient-4.2.3.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpcore-4.2.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\jcl-over-slf4j-1.6.4.jar;C:\Users\g

bad_sv4_redundant                     1041.0 ms  consistency=passed, acyclicity=passed, reachability=passed, redundancy=failed, subsumption=passed


* Owlready * Adding relation bad_sv5_subject.student_advanced satisfies bad_sv5_subject.p_subj_group_sub_group
bad_sv5_subject                       1042.4 ms  consistency=passed, acyclicity=passed, reachability=passed, redundancy=passed, subsumption=failed


* Owlready2 * Pellet took 1.0228972434997559 seconds
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)


In [4]:
pairs = [(dict(report.properties), spec['expected']) for spec, report, _ in runs]
matrices = aggregate_by_property(pairs)
macro = macro_average(matrices)

md_table = format_markdown_table(matrices)
csv_text = format_csv(matrices)

print(md_table)
print()
print(f"macro-avg: precision={macro['precision']:.3f}  recall={macro['recall']:.3f}  f1={macro['f1']:.3f}")

| Свойство | TP | FP | FN | TN | Precision | Recall | F1 | Support |
|---|---|---|---|---|---|---|---|---|
| consistency | 1 | 0 | 0 | 7 | 1.000 | 1.000 | 1.000 | 1 |
| acyclicity | 2 | 0 | 0 | 5 | 1.000 | 1.000 | 1.000 | 2 |
| reachability | 3 | 0 | 0 | 3 | 1.000 | 1.000 | 1.000 | 3 |
| redundancy | 1 | 0 | 0 | 1 | 1.000 | 1.000 | 1.000 | 1 |
| subsumption | 1 | 0 | 0 | 1 | 1.000 | 1.000 | 1.000 | 1 |
| **macro-avg** | . | . | . | . | 1.000 | 1.000 | 1.000 | . |

macro-avg: precision=1.000  recall=1.000  f1=1.000


In [5]:
RESULTS_DIR.mkdir(exist_ok=True)
PZ_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

durations_csv_lines = ['scenario,duration_ms']
for spec, _, duration_ms in runs:
    durations_csv_lines.append(f"{spec['name']},{duration_ms:.1f}")
durations_csv = '\n'.join(durations_csv_lines) + '\n'

for target_dir in (RESULTS_DIR, PZ_FIGURES_DIR):
    (target_dir / 'pilot_day1.csv').write_text(csv_text, encoding='utf-8')
    (target_dir / 'pilot_day1.md').write_text(md_table + '\n', encoding='utf-8')
    (target_dir / 'pilot_day1_durations.csv').write_text(durations_csv, encoding='utf-8')

print('Экспортировано в:')
print(f'  {RESULTS_DIR}')
print(f'  {PZ_FIGURES_DIR}')

Экспортировано в:
  C:\Documents\itmo\vkr\vkr-access-control\experiments\exp1_verification\results
  C:\Documents\itmo\vkr\vkr-access-control\pz\figures\exp1
